# SafeWatch — 차량 검출 파인튜닝 (단일 클래스)데이터셋의 25개 클래스 중 차량 4종만 `vehicle` 단일 클래스로 통합하여 학습한다.**실행**: 1 → 2 → (3) → 4 → 5 → 6 순서로. 2번에서 백업이 복원되면 3번은 건너뛴다.**세션이 끊기면**: 1 → 2 실행 후, 학습 중이었으면 맨 아래 재개 셀로.

## 1. 환경 준비마운트에서 `credential propagation was unsuccessful` 오류가 나면 런타임 재시작 후 다시 실행한다.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader

from google.colab import drive
drive.mount('/content/drive')

!pip install -q ultralytics

from pathlib import Path
from ultralytics import YOLO
import ultralytics
print('ultralytics', ultralytics.__version__)

# ── 경로 ──────────────────────────────────────────────
DRIVE_ROOT = Path('/content/drive/MyDrive/SafeWatch')
DRIVE_DATA = DRIVE_ROOT / '도로주행영상'
TRAIN_DIR  = DRIVE_DATA / 'Training'
VALID_DIR  = DRIVE_DATA / 'Validation'
RUNS_DIR   = DRIVE_ROOT / 'runs'

OUT    = Path('/content/yolo_vehicle')      # 최종 학습 데이터
RAW_TR = Path('/content/train_raw')
RAW_VA = Path('/content/val_raw')
BACKUP = DRIVE_ROOT / 'yolo_vehicle.tar'

# ── 설정 ──────────────────────────────────────────────
VEHICLE  = {'Vehicle_Car', 'Vehicle_Bus', 'Vehicle_Motorcycle', 'Vehicle_Unknown'}
N_TRAIN  = 6000     # 원천 zip당 이미지 수
N_VALID  = 500
RUN_NAME = 'vehicle_v1'
IMGSZ, EPOCHS, BATCH = 320, 100, 32

RUNS_DIR.mkdir(parents=True, exist_ok=True)
assert DRIVE_DATA.exists(), f'경로 없음: {DRIVE_DATA}'
print('준비 완료 →', DRIVE_DATA)

## 2. 백업 확인이전에 만들어 둔 학습 데이터가 드라이브에 있으면 복원한다. 복원되면 3번 셀은 건너뛴다.

In [ ]:
import subprocess, time

READY = False
if BACKUP.exists():
    print(f'백업 발견 [{BACKUP.stat().st_size/1e9:.2f} GB] → 복원 중')
    t0 = time.time()
    subprocess.run(['tar', '-xf', str(BACKUP), '-C', '/content'], check=True)
    READY = (OUT / 'data.yaml').exists()
    print(f'복원 {"성공" if READY else "실패"} ({time.time()-t0:.0f}s)')
else:
    print('백업 없음 → 3번 셀부터 진행')

if READY:
    DATA = OUT / 'data.yaml'
    n_tr = len(list((OUT/'images'/'train').glob('*')))
    n_va = len(list((OUT/'images'/'val').glob('*')))
    print(f'train {n_tr}장 / val {n_va}장')
    print('→ 3번 건너뛰고 4번으로')

## 3. 원본 해제 (백업이 없을 때만)`bb`(바운딩박스) 폴더만 사용한다. 라벨은 전부, 이미지는 설정한 개수만 꺼낸다.zip 종류는 이름이 아니라 **내용으로 판별**한다. 한글 파일명이 드라이브를 거치며 자모 분리되는 경우가 있어서다.

In [ ]:
import zipfile, random, shutil, time

if READY:
    print('백업에서 복원됨 — 건너뜀')
else:
    def unpack(src_dir: Path, dest: Path, n_img: int):
        shutil.rmtree(dest, ignore_errors=True)
        for z in sorted(src_dir.rglob('*.zip')):
            with zipfile.ZipFile(z) as zf:
                bb  = [x for x in zf.namelist()
                       if x.startswith('bb/') and not x.endswith('/')]
                xml = [x for x in bb if x.lower().endswith('.xml')]
                img = [x for x in bb if x.lower().endswith(('.jpg', '.png'))]

                if xml:                       # 라벨 zip → 전부
                    pick = xml
                else:                         # 원천 zip → 일부
                    random.seed(42)
                    pick = sorted(random.sample(img, min(n_img, len(img))))

                print(f'  {z.name}: bb {len(bb)}개 중 {len(pick)}개')
                for x in pick:
                    zf.extract(x, dest)

    t0 = time.time()
    print('[Training]');   unpack(TRAIN_DIR / 'bb', RAW_TR, N_TRAIN)
    print('[Validation]'); unpack(VALID_DIR,        RAW_VA, N_VALID)

    for tag, d in [('train', RAW_TR), ('val', RAW_VA)]:
        print(f'{tag}: 이미지 {len(list(d.rglob("*.jpg")))} / XML {len(list(d.rglob("*.xml")))}')
    print(f'\n완료 ({time.time()-t0:.0f}s)')
    !df -h /content | tail -1

## 4. 단일 클래스 변환XML(Pascal VOC)을 YOLO 형식으로 바꾸면서 차량 4종을 `vehicle` 하나로 통합한다.보행자·신호등·노면표시는 본 과제에 불필요하므로 제외한다.

In [ ]:
import xml.etree.ElementTree as ET
import re, shutil, yaml

if READY:
    print('백업에서 복원됨 — 건너뜀')
else:
    norm = lambda s: re.sub(r'_v\d+_\d+$', '', Path(s).stem)
    shutil.rmtree(OUT, ignore_errors=True)

    def convert(raw: Path, split: str):
        (OUT/'images'/split).mkdir(parents=True, exist_ok=True)
        (OUT/'labels'/split).mkdir(parents=True, exist_ok=True)
        xmap = {norm(x): x for x in raw.rglob('*.xml')}
        imgs = list(raw.rglob('*.jpg')) + list(raw.rglob('*.png'))
        hit  = sum(1 for p in imgs if p.stem in xmap)
        print(f'  {split}: 이미지 {len(imgs)} / 매칭 {hit}')

        ok = box = 0
        for ip in imgs:
            xp = xmap.get(ip.stem)
            if not xp: continue
            try:
                r = ET.parse(xp).getroot()
                W = float(r.findtext('size/width')); H = float(r.findtext('size/height'))
                if W <= 0 or H <= 0: continue
                lines = []
                for o in r.iter('object'):
                    if (o.findtext('name') or '').strip() not in VEHICLE: continue
                    b = o.find('bndbox')
                    x1 = max(0, min(float(b.findtext('xmin')), W))
                    y1 = max(0, min(float(b.findtext('ymin')), H))
                    x2 = max(0, min(float(b.findtext('xmax')), W))
                    y2 = max(0, min(float(b.findtext('ymax')), H))
                    if x2-x1 < 1 or y2-y1 < 1: continue
                    lines.append(f'0 {((x1+x2)/2)/W:.6f} {((y1+y2)/2)/H:.6f} '
                                 f'{(x2-x1)/W:.6f} {(y2-y1)/H:.6f}')
                if not lines: continue
                shutil.copy(ip, OUT/'images'/split/ip.name)
                (OUT/'labels'/split/f'{ip.stem}.txt').write_text('\n'.join(lines))
                ok += 1; box += len(lines)
            except Exception:
                pass
        print(f'     → {ok}장 / 박스 {box}개 (장당 {box/max(ok,1):.1f})')
        return ok

    n_tr = convert(RAW_TR, 'train')
    n_va = convert(RAW_VA, 'val')
    assert n_tr > 0 and n_va > 0, '변환 결과가 비어 있음'

    DATA = OUT / 'data.yaml'
    yaml.safe_dump({'path': str(OUT), 'train': 'images/train',
                    'val': 'images/val', 'names': {0: 'vehicle'}},
                   open(DATA, 'w'), allow_unicode=True)
    print('\nDATA =', DATA)

## 5. 백업**학습 전에 반드시 실행한다.** 세션이 끊기면 `/content`가 통째로 사라지므로,여기서 백업해 두면 다음 세션은 2번 셀에서 복원만 하면 된다.

In [ ]:
import subprocess, time

if BACKUP.exists():
    print('백업 이미 있음 — 건너뜀')
else:
    t0 = time.time()
    subprocess.run(['tar', '-cf', '/content/yolo_vehicle.tar',
                    '-C', '/content', 'yolo_vehicle'], check=True)
    subprocess.run(['cp', '/content/yolo_vehicle.tar', str(BACKUP)], check=True)
    print(f'백업 완료 ({time.time()-t0:.0f}s) → {BACKUP}')

## 6. 학습체크포인트는 드라이브에 저장되므로 세션이 끊겨도 남는다.

In [ ]:
model = YOLO('yolov8n.pt')

model.train(
    data=str(DATA),
    epochs=EPOCHS,
    imgsz=IMGSZ,
    batch=BATCH,
    patience=25,
    save_period=5,
    workers=2,
    project=str(RUNS_DIR),
    name=RUN_NAME,
    exist_ok=True,
    device=0,
    plots=True,
)
print('학습 완료')

## 7. 평가비교 기준 — 25개 클래스로 학습했을 때의 **차량 4종 평균**은 다음과 같았다.| | P | R | mAP50 ||---|---|---|---|| 25클래스 모델의 차량 평균 | 0.602 | 0.491 | 0.500 || 25클래스 전체 평균 | 0.459 | 0.190 | 0.193 |아래 결과가 첫 번째 행보다 높아야 단일 클래스 통합이 효과가 있었다는 뜻이다.

In [ ]:
best = YOLO(str(RUNS_DIR / RUN_NAME / 'weights' / 'best.pt'))
m = best.val(data=str(DATA), imgsz=IMGSZ)

print()
print(f'Precision : {m.box.mp:.4f}')
print(f'Recall    : {m.box.mr:.4f}')
print(f'mAP50     : {m.box.map50:.4f}')
print(f'mAP50-95  : {m.box.map:.4f}')
print()
print('참고 — 25클래스 모델 차량 평균: P 0.602 / R 0.491 / mAP50 0.500')

신뢰도 임계값에 따른 변화. 본 시스템은 복수 지표 결합 판정으로 오탐을 한 번 더 거르므로,검출 단계에서는 Recall을 조금 더 확보하는 쪽이 유리할 수 있다.

In [ ]:
print('conf     Precision  Recall   mAP50')
for c in [0.001, 0.01, 0.05, 0.1, 0.25]:
    r = best.val(data=str(DATA), imgsz=IMGSZ, conf=c, verbose=False)
    print(f'{c:<8} {r.box.mp:.3f}      {r.box.mr:.3f}    {r.box.map50:.3f}')

## 8. 먼 차량 검출률
차 바닥 높이별 Recall — 실행 입력 320 vs 640. 차선 노트북 v2 29번 셀(앞차 절반이 화면 높이 0.58보다 멂)의 후속.

In [ ]:
# 8. 먼 차량 검출률 — 차 바닥 높이별 Recall (차선 노트북 29번과 같은 높이 구간)
# 필요: 1·2번 셀 (OUT = 학습 데이터, RUNS_DIR·RUN_NAME). GPU 권장, 몇 분.
#
# 왜: 차선 노트북 29번 셀 — 앞차의 약 절반은 차 바닥이 화면 높이 0.58보다 먼 곳에 있다. 차선을 먼 곳까지 잡아도
#     차량을 못 찾으면 소용없다 → 먼 차를 얼마나 찾는지 먼저 잰다.
# 보는 것: 실행 조건(Pi 입력 320) vs 640 — 640에서 크게 오르면 원인은 해상도(학습으로는 한계), 비슷하게 낮으면 학습 문제.
# 기준: 정답 박스(val 라벨) 중 가로 가운데 70% 안(실행 detection.center_region_ratio)인 것만. IoU 0.5 이상 예측이 있으면 찾음.
#       conf 0.1 = 실행 detection.conf_threshold(기존 트랙 잇기), 0.5 = 새 트랙 시작(tracking.track_high_thresh)
import numpy as np, time
from collections import defaultdict

W8 = {'파인튜닝': RUNS_DIR / RUN_NAME / 'weights' / 'best.pt'}   # 실행 가중치(yolov8n-safewatch)와 같은 것
IMGSZ8  = [320, 640]
CONF8   = [0.1, 0.5]
CENTER8 = 0.7
BINS8   = [0.0, 0.42, 0.50, 0.54, 0.58, 0.62, 0.66, 0.70, 0.80, 1.01]
IOU8    = 0.5

imgs8 = sorted((OUT / 'images' / 'val').glob('*'))
if not imgs8:
    raise SystemExit(f'{OUT}/images/val 비었음 — 2번 셀(복원) 먼저')


def gt8(ip):
    lp = OUT / 'labels' / 'val' / f'{ip.stem}.txt'
    if not lp.exists():
        return np.zeros((0, 4))
    a = np.loadtxt(lp, ndmin=2)[:, 1:]                   # cx cy w h (0~1)
    b = np.c_[a[:, 0] - a[:, 2] / 2, a[:, 1] - a[:, 3] / 2, a[:, 0] + a[:, 2] / 2, a[:, 1] + a[:, 3] / 2]
    cx = a[:, 0]
    return b[np.abs(cx - 0.5) <= CENTER8 / 2]


def iou8(g, p):
    x1, y1 = np.maximum(g[0], p[:, 0]), np.maximum(g[1], p[:, 1])
    x2, y2 = np.minimum(g[2], p[:, 2]), np.minimum(g[3], p[:, 3])
    inter = np.clip(x2 - x1, 0, None) * np.clip(y2 - y1, 0, None)
    return inter / ((g[2] - g[0]) * (g[3] - g[1]) + (p[:, 2] - p[:, 0]) * (p[:, 3] - p[:, 1]) - inter)


GT8 = {ip: gt8(ip) for ip in imgs8}
n_gt = sum(len(v) for v in GT8.values())
print(f'val {len(imgs8)}장 · 가운데 {int(CENTER8 * 100)}% 안 정답 차량 {n_gt}대')

HIT8 = {}                                              # (모델, imgsz, conf) → [(바닥 높이, 박스 높이, 찾음)]
for name, wp in W8.items():
    model = YOLO(str(wp))
    for sz in IMGSZ8:
        t0, rows = time.time(), {c: [] for c in CONF8}
        for i in range(0, len(imgs8), 32):
            batch = imgs8[i:i + 32]
            rs = model.predict([str(p) for p in batch], imgsz=sz, conf=min(CONF8), verbose=False, device=0)
            for ip, r in zip(batch, rs):
                g = GT8[ip]
                if len(g) == 0:
                    continue
                p = r.boxes.xyxyn.cpu().numpy(); s = r.boxes.conf.cpu().numpy()
                for c in CONF8:
                    pc, used = p[s >= c], set()
                    for gb in sorted(g, key=lambda b: -(b[3] - b[1])):      # 큰(가까운) 차부터 짝짓기
                        hit = False
                        if len(pc):
                            ious = iou8(gb, pc)
                            for j in np.argsort(-ious):
                                if ious[j] < IOU8:
                                    break
                                if j not in used:
                                    used.add(j); hit = True
                                    break
                        rows[c].append((gb[3], gb[3] - gb[1], hit))
        for c in CONF8:
            HIT8[(name, sz, c)] = rows[c]
        print(f'  {name} imgsz {sz}: {time.time() - t0:.0f}초')

print('\n' + '═' * 110)
print('칸 = Recall (찾은 대수/정답 대수)     높이 = 차 바닥(화면 높이 비율, 위 = 멀리)  ·  차선 노트북 29번: 지금 offset을 재는 건 0.58 이상')
print('═' * 110)
keys = [k for k in HIT8]
print(f'  {"차 바닥 높이":<12}{"대수":>6}  ' + ''.join(f'{f"{k[0]} {k[1]} conf{k[2]}":>22}' for k in keys))
groups = [(f'{a:.2f}~{min(b, 1):.2f}', lambda y, a=a, b=b: a <= y < b) for a, b in zip(BINS8[:-1], BINS8[1:])]
groups += [('0.58 위(먼 차)', lambda y: y < 0.58), ('0.58 이상', lambda y: y >= 0.58), ('전체', lambda y: True)]
for gname, cond in groups:
    cells, n = [], 0
    for k in keys:
        rs = [r for r in HIT8[k] if cond(r[0])]
        n = len(rs)
        cells.append(f'{(sum(r[2] for r in rs) / n if n else float("nan")):.2f} ({sum(r[2] for r in rs)}/{n})' if n else '-')
    print(f'  {gname:<12}{n:>6}  ' + ''.join(f'{c:>22}' for c in cells))

print('\n박스 높이별 (화면 높이 대비, 작을수록 멀다)')
HB8 = [0, 0.02, 0.04, 0.08, 1.01]
for a, b in zip(HB8[:-1], HB8[1:]):
    cells, n = [], 0
    for k in keys:
        rs = [r for r in HIT8[k] if a <= r[1] < b]
        n = len(rs)
        cells.append(f'{sum(r[2] for r in rs) / n:.2f} ({sum(r[2] for r in rs)}/{n})' if n else '-')
    print(f'  {a:.2f}~{min(b, 1):.2f}    {n:>6}  ' + ''.join(f'{c:>22}' for c in cells))

---## 재개 (학습 중 세션이 끊긴 경우)1 → 2번 셀을 실행한 뒤 아래를 실행한다. 중단된 에폭부터 이어서 학습한다.

In [ ]:
last = RUNS_DIR / RUN_NAME / 'weights' / 'last.pt'
assert last.exists(), f'체크포인트 없음: {last}'

model = YOLO(str(last))
model.train(resume=True)